# 04 — Feast Feature Store

- Prepare Feast-formatted Parquet (done by `etl/gold.py`)
- `feast apply` (register feature views)
- `feast materialize` (push offline → online store)
- Demonstrate `get_historical_features()` and `get_online_features()`

**Prerequisites:** Gold layer built (NB03). Run from project root.

In [ ]:
import sys
sys.path.insert(0, '..')

import subprocess
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from feast import FeatureStore

FEATURE_STORE_REPO = Path('../feature_store').resolve()

## 4.1 Feast apply (register schema)

In [ ]:
result = subprocess.run(['feast', 'apply'], cwd=str(FEATURE_STORE_REPO), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 4.2 Materialize offline → online store

In [ ]:
store = FeatureStore(repo_path=str(FEATURE_STORE_REPO))

start = datetime(2018, 1, 1, tzinfo=timezone.utc)
end   = datetime.now(timezone.utc)

print(f'Materializing {start.date()} → {end.date()}...')
store.materialize(start_date=start, end_date=end)
print('Materialization complete.')

## 4.3 Historical features (point-in-time join)

This is how training features should be retrieved when retraining.

In [ ]:
# Build entity DataFrame (timestamps we want features for)
entity_df = pd.DataFrame({
    'market_zone':        ['DE_LU'] * 5,
    'prediction_ts_utc':  [
        '2023-01-15T12:00:00Z',
        '2023-03-10T08:00:00Z',
        '2023-06-21T14:00:00Z',
        '2023-09-05T18:00:00Z',
        '2023-12-25T20:00:00Z',
    ],
    'event_timestamp': pd.to_datetime([
        '2023-01-15T12:00:00Z',
        '2023-03-10T08:00:00Z',
        '2023-06-21T14:00:00Z',
        '2023-09-05T18:00:00Z',
        '2023-12-25T20:00:00Z',
    ], utc=True)
})

feature_vector = store.get_historical_features(
    entity_df=entity_df,
    features=[
        'price_lag_features:price_lag_24h',
        'price_lag_features:price_roll_mean_7d',
        'calendar_features:hour_of_day',
        'weather_features:temperature_2m',
    ]
).to_df()

print(feature_vector)

## 4.4 Online features (serving-time lookup)

In [ ]:
online_features = store.get_online_features(
    features=[
        'price_lag_features:price_lag_1h',
        'price_lag_features:price_lag_24h',
        'price_lag_features:price_roll_mean_7d',
        'calendar_features:hour_of_day',
        'calendar_features:is_weekend',
        'weather_features:temperature_2m',
    ],
    entity_rows=[{
        'market_zone': 'DE_LU',
        'prediction_ts_utc': '2023-12-25T20:00:00Z'
    }]
).to_df()

print('Online feature vector:')
print(online_features.T)